# SIDER EDA: Complete Analysis of All Files

Comprehensive analysis of SIDER dataset with df.head() for all files including drug_names, drug_atc, all SE files, and indications.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('bmh')
sider_path = r'D:\ADE DATASET DOWNLOAD\SIDER'
eda_path = r'D:\ADE DATASET DOWNLOAD\EDA'

## 1. Data Loading Function

In [ ]:
def load_sider_tsv(filename, columns):
    path = os.path.join(sider_path, filename)
    if not os.path.exists(path): 
        print(f"File not found: {filename}")
        return pd.DataFrame()
    
    compression = 'gzip' if filename.endswith('.gz') else None
    return pd.read_csv(path, sep='\t', names=columns, compression=compression, low_memory=False)

## 2. Drug Names Mapping

In [ ]:
drug_names = load_sider_tsv('drug_names.tsv', ['stitch_id', 'drug_name'])
print(f"Total Drug Names: {len(drug_names):,}")
display(drug_names.head(5))

## 3. Drug ATC Classification

In [ ]:
drug_atc = load_sider_tsv('drug_atc.tsv', ['stitch_id', 'atc_code'])print(f"Total Drug-ATC Mappings: {len(drug_atc):,}")display(drug_atc.head(5))if not drug_atc.empty and 'atc_code' in drug_atc.columns:    drug_atc['atc_level1'] = drug_atc['atc_code'].str[0]        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # ATC Level 1 distribution    atc_dist = drug_atc['atc_level1'].value_counts()    sns.barplot(x=atc_dist.index, y=atc_dist.values, ax=axes[0,0], palette='viridis')    axes[0,0].set_title('Drug Distribution by ATC Level 1 Class', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('ATC Level 1 Code')    axes[0,0].set_ylabel('Count')        # Pie chart    axes[0,1].pie(atc_dist.values, labels=atc_dist.index, autopct='%1.1f%%', startangle=90)    axes[0,1].set_title('Proportional ATC Level 1 Distribution', fontsize=14, fontweight='bold')        # Drugs per ATC code    atc_per_drug = drug_atc.groupby('stitch_id')['atc_code'].nunique()    sns.histplot(atc_per_drug, bins=20, kde=True, ax=axes[1,0], color='teal')    axes[1,0].set_title('ATC Codes per Drug', fontsize=14, fontweight='bold')    axes[1,0].set_xlabel('Number of ATC Codes')    axes[1,0].axvline(atc_per_drug.median(), color='red', linestyle='--', label=f'Median: {atc_per_drug.median():.0f}')    axes[1,0].legend()        # Summary stats table    atc_summary = pd.DataFrame({        'Total Mappings': [len(drug_atc)],        'Unique Drugs': [drug_atc['stitch_id'].nunique()],        'Unique ATC Codes': [drug_atc['atc_code'].nunique()],        'Avg ATC/Drug': [atc_per_drug.mean()]    })    axes[1,1].axis('off')    table = axes[1,1].table(cellText=atc_summary.T.values,                            rowLabels=atc_summary.T.index,                           colLabels=['Value'],                           cellLoc='center', loc='center')    table.auto_set_font_size(False)    table.set_fontsize(11)    table.scale(1, 3)    axes[1,1].set_title('ATC Classification Summary', fontsize=14, fontweight='bold')        plt.tight_layout()    plt.show()        print(f"\nATC Statistics:")    print(f"  Unique ATC codes: {drug_atc['atc_code'].nunique():,}")    print(f"  Drugs with multiple ATC codes: {(atc_per_drug > 1).sum():,}")

## 4. All Side Effects File

In [ ]:
meddra_se = load_sider_tsv('meddra_all_se.tsv.gz',                            ['stitch_id_1', 'stitch_id_2', 'umls_id_label', 'meddra_type', 'umls_id_meddra', 'side_effect_name'])print(f"Total Side Effect Entries: {len(meddra_se):,}")display(meddra_se.head(5))if not meddra_se.empty and 'side_effect_name' in meddra_se.columns:    top_se = meddra_se['side_effect_name'].value_counts().head(20)    se_per_drug = meddra_se.groupby('stitch_id_2')['side_effect_name'].nunique()        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Top side effects    sns.barplot(x=top_se.values, y=top_se.index, ax=axes[0,0], palette='crest')    axes[0,0].set_title('Top 20 Most Prevalent Side Effects on Drug Labels', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Number of Drug-SE Pairs')        # Side effects per drug distribution    sns.histplot(se_per_drug, bins=40, kde=True, ax=axes[0,1], color='crimson')    axes[0,1].set_title('Side Effects per Drug Distribution', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Side Effects')    axes[0,1].axvline(se_per_drug.median(), color='yellow', linestyle='--', label=f'Median: {se_per_drug.median():.0f}')    axes[0,1].legend()        # SE burden categories    se_categories = pd.cut(se_per_drug,                           bins=[0, 10, 50, 100, 200, float('inf')],                          labels=['1-10', '11-50', '51-100', '101-200', '200+'])    cat_counts = se_categories.value_counts().sort_index()    sns.barplot(x=cat_counts.index, y=cat_counts.values, ax=axes[1,0], palette='RdYlGn_r')    axes[1,0].set_title('Drug SE Burden Categories', fontsize=14, fontweight='bold')    axes[1,0].set_xlabel('Number of Side Effects')    axes[1,0].set_ylabel('Number of Drugs')        # MedDRA type distribution    if 'meddra_type' in meddra_se.columns:        meddra_type_dist = meddra_se['meddra_type'].value_counts()        axes[1,1].pie(meddra_type_dist.values, labels=meddra_type_dist.index,                      autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))        axes[1,1].set_title('MedDRA Type Distribution', fontsize=14, fontweight='bold')        plt.tight_layout()    plt.show()        print(f"\nSide Effect Statistics:")    print(f"  Total unique side effects: {meddra_se['side_effect_name'].nunique():,}")    print(f"  Total unique drugs: {meddra_se['stitch_id_2'].nunique():,}")    print(f"  Average SEs per drug: {se_per_drug.mean():.2f}")    print(f"  Max SEs for one drug: {se_per_drug.max()}")

## 5. Frequency Data

In [ ]:
meddra_freq = load_sider_tsv('meddra_freq.tsv.gz',                              ['stitch_id_1', 'stitch_id_2', 'umls_id_label', 'frequency_description',                               'frequency_lower', 'frequency_upper', 'frequency_meddra', 'umls_id_meddra', 'side_effect_name'])print(f"Total Frequency Entries: {len(meddra_freq):,}")display(meddra_freq.head(5))if not meddra_freq.empty and 'frequency_description' in meddra_freq.columns:    top_freqs = meddra_freq['frequency_description'].value_counts().head(15)        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Frequency keywords    sns.barplot(x=top_freqs.values, y=top_freqs.index, ax=axes[0,0], palette='flare')    axes[0,0].set_title('Top ADR Frequency Keywords on Drug Labels', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Count')        # Frequency bounds distribution    if 'frequency_lower' in meddra_freq.columns and 'frequency_upper' in meddra_freq.columns:        meddra_freq['frequency_lower'] = pd.to_numeric(meddra_freq['frequency_lower'], errors='coerce')        meddra_freq['frequency_upper'] = pd.to_numeric(meddra_freq['frequency_upper'], errors='coerce')        valid_lower = meddra_freq['frequency_lower'].dropna()        valid_upper = meddra_freq['frequency_upper'].dropna()                if len(valid_lower) > 0:            sns.histplot(valid_lower * 100, bins=30, kde=True, ax=axes[0,1], color='orange', label='Lower Bound')            axes[0,1].set_title('Frequency Lower Bound Distribution', fontsize=14, fontweight='bold')            axes[0,1].set_xlabel('Frequency (%)')            axes[0,1].set_xlim(0, 100)        # Drugs with frequency info    drugs_with_freq = meddra_freq['stitch_id_2'].nunique()    freq_per_drug = meddra_freq.groupby('stitch_id_2').size()    sns.histplot(freq_per_drug, bins=30, kde=True, ax=axes[1,0], color='purple')    axes[1,0].set_title('Frequency Entries per Drug', fontsize=14, fontweight='bold')    axes[1,0].set_xlabel('Number of Frequency Records')    axes[1,0].axvline(freq_per_drug.median(), color='yellow', linestyle='--', label=f'Median: {freq_per_drug.median():.0f}')    axes[1,0].legend()        # Summary table    freq_summary = pd.DataFrame({        'Freq Records': [len(meddra_freq)],        'Drugs with Freq': [drugs_with_freq],        'SE with Freq': [meddra_freq['side_effect_name'].nunique()],        'Avg Freq/Drug': [freq_per_drug.mean()]    })    axes[1,1].axis('off')    table = axes[1,1].table(cellText=freq_summary.T.values,                            rowLabels=freq_summary.T.index,                           colLabels=['Value'],                           cellLoc='center', loc='center')    table.auto_set_font_size(False)    table.set_fontsize(11)    table.scale(1, 3)    axes[1,1].set_title('Frequency Data Summary', fontsize=14, fontweight='bold')        plt.tight_layout()    plt.show()        print(f"\nFrequency Statistics:")    print(f"  Drugs with frequency data: {drugs_with_freq:,}")    print(f"  SEs with frequency data: {meddra_freq['side_effect_name'].nunique():,}")

## 6. All Label Side Effects

In [ ]:
label_se = load_sider_tsv('meddra_all_label_se.tsv.gz',
                          ['stitch_id_1', 'stitch_id_2', 'umls_id_label', 'meddra_concept_type', 'umls_id_meddra', 'side_effect_name'])
print(f"Total Label SE Entries: {len(label_se):,}")
display(label_se.head(5))

## 7. All Indications

In [ ]:
indications = load_sider_tsv('meddra_all_indications.tsv.gz',                             ['stitch_id_1', 'stitch_id_2', 'umls_id_label', 'method_of_detection', 'concept_name', 'meddra_concept_type', 'umls_id_meddra', 'meddra_concept_name'])print(f"Total Indication Entries: {len(indications):,}")display(indications.head(5))if not indications.empty and 'meddra_concept_name' in indications.columns:    top_indications = indications['meddra_concept_name'].value_counts().head(20)    ind_per_drug = indications.groupby('stitch_id_2')['meddra_concept_name'].nunique()        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Top indications    sns.barplot(x=top_indications.values, y=top_indications.index, ax=axes[0,0], palette='rocket')    axes[0,0].set_title('Top 20 Drug Indications', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Number of Drugs')        # Indications per drug    sns.histplot(ind_per_drug, bins=30, kde=True, ax=axes[0,1], color='darkblue')    axes[0,1].set_title('Indications per Drug Distribution', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Indications')    axes[0,1].axvline(ind_per_drug.median(), color='red', linestyle='--', label=f'Median: {ind_per_drug.median():.0f}')    axes[0,1].legend()        # Method of detection    if 'method_of_detection' in indications.columns:        method_dist = indications['method_of_detection'].value_counts()        axes[1,0].pie(method_dist.values, labels=method_dist.index,                      autopct='%1.1f%%', startangle=90, colors=sns.color_palette('Set2'))        axes[1,0].set_title('Method of Detection Distribution', fontsize=14, fontweight='bold')        # Indication burden categories    ind_categories = pd.cut(ind_per_drug,                            bins=[0, 1, 3, 5, 10, float('inf')],                           labels=['1', '2-3', '4-5', '6-10', '10+'])    cat_counts = ind_categories.value_counts().sort_index()    sns.barplot(x=cat_counts.index, y=cat_counts.values, ax=axes[1,1], palette='cividis')    axes[1,1].set_title('Drug Indication Burden Categories', fontsize=14, fontweight='bold')    axes[1,1].set_xlabel('Number of Indications')    axes[1,1].set_ylabel('Number of Drugs')        plt.tight_layout()    plt.show()        print(f"\nIndication Statistics:")    print(f"  Total unique indications: {indications['meddra_concept_name'].nunique():,}")    print(f"  Total unique drugs: {indications['stitch_id_2'].nunique():,}")    print(f"  Average indications per drug: {ind_per_drug.mean():.2f}")

## 8. MedDRA File

In [ ]:
meddra = load_sider_tsv('meddra.tsv.gz',
                        ['umls_cui_from_label', 'meddra_concept_type', 'umls_cui_from_meddra', 'side_effect_name'])
print(f"Total MedDRA Mappings: {len(meddra):,}")
display(meddra.head(5))

if not meddra.empty and 'meddra_concept_type' in meddra.columns:
    meddra_types = meddra['meddra_concept_type'].value_counts()
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=meddra_types.index, y=meddra_types.values, palette='Set2')
    plt.title('MedDRA Concept Type Distribution')
    plt.tight_layout()
    plt.show()

## 9. Drug Complexity Analysis

In [ ]:
if not meddra_se.empty and not drug_names.empty:    drug_se_counts = meddra_se.groupby('stitch_id_2')['side_effect_name'].nunique().reset_index(name='adr_count')    drug_se_counts = pd.merge(drug_se_counts, drug_names, left_on='stitch_id_2', right_on='stitch_id')        top_20_drugs = drug_se_counts.sort_values('adr_count', ascending=False).head(20)        fig, axes = plt.subplots(2, 2, figsize=(18, 12))        # Top drugs by SE count    sns.barplot(data=top_20_drugs, x='adr_count', y='drug_name', ax=axes[0,0], palette='magma')    axes[0,0].set_title('Top 20 Drugs by Unique Side Effect Count (Label Complexity)', fontsize=14, fontweight='bold')    axes[0,0].set_xlabel('Number of Unique Side Effects')        # Overall distribution    sns.histplot(drug_se_counts['adr_count'], bins=40, kde=True, ax=axes[0,1], color='darkred')    axes[0,1].set_title('Distribution of SE Count Across All Drugs', fontsize=14, fontweight='bold')    axes[0,1].set_xlabel('Number of Side Effects')    axes[0,1].axvline(drug_se_counts['adr_count'].median(), color='yellow', linestyle='--',                      label=f'Median: {drug_se_counts["adr_count"].median():.0f}')    axes[0,1].legend()        # Boxplot    sns.boxplot(y=drug_se_counts['adr_count'], ax=axes[1,0], color='coral')    axes[1,0].set_title('Drug Complexity Boxplot', fontsize=14, fontweight='bold')    axes[1,0].set_ylabel('Number of Side Effects')        # Bottom 20 drugs    bottom_20_drugs = drug_se_counts.sort_values('adr_count', ascending=True).head(20)    sns.barplot(data=bottom_20_drugs, x='adr_count', y='drug_name', ax=axes[1,1], palette='winter')    axes[1,1].set_title('Bottom 20 Drugs by SE Count (Simplest Labels)', fontsize=14, fontweight='bold')    axes[1,1].set_xlabel('Number of Unique Side Effects')        plt.tight_layout()    plt.show()        print(f"\nDrug Complexity Statistics:")    print(f"  Most complex drug: {drug_se_counts.loc[drug_se_counts['adr_count'].idxmax(), 'drug_name']} ({drug_se_counts['adr_count'].max()} SEs)")    print(f"  Simplest drug: {drug_se_counts.loc[drug_se_counts['adr_count'].idxmin(), 'drug_name']} ({drug_se_counts['adr_count'].min()} SEs)")    print(f"  Average SEs per drug: {drug_se_counts['adr_count'].mean():.2f}")

## 10. Summary Report

In [ ]:
summary = {    'File': ['drug_names', 'drug_atc', 'meddra_all_se', 'meddra_freq', 'meddra_all_label_se', 'meddra_all_indications', 'meddra'],    'Records': [len(drug_names), len(drug_atc), len(meddra_se), len(meddra_freq), len(label_se), len(indications), len(meddra)]}summary_df = pd.DataFrame(summary)display(summary_df)fig, axes = plt.subplots(1, 2, figsize=(18, 7))# Bar chart with annotationssns.barplot(data=summary_df, x='Records', y='File', ax=axes[0], palette='mako')axes[0].set_title('SIDER Dataset File Coverage', fontsize=14, fontweight='bold')axes[0].set_xlabel('Number of Records')for i, v in enumerate(summary_df['Records']):    axes[0].text(v, i, f' {v:,}', va='center', fontsize=9)# Pie chartsummary_df_sorted = summary_df.sort_values('Records', ascending=False)axes[1].pie(summary_df_sorted['Records'], labels=summary_df_sorted['File'],            autopct=lambda pct: f'{pct:.1f}%' if pct > 2 else '',           startangle=90, colors=sns.color_palette('mako', len(summary_df)))axes[1].set_title('Proportional Data Distribution', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()print("\n" + "="*60)print("SIDER COMPREHENSIVE ANALYSIS COMPLETE")print("="*60)print(f"Total Records Analyzed: {summary_df['Records'].sum():,}")print(f"Total Files Covered: {len(summary_df)}")print("="*60)